# The categories $\mathbf{Fact}$, $\mathbf{Ref}$, and spans

A companion to `tract.fact_morphism`, `tract.ref_morphism`, and `tract.spans`, part of
the code accompanying *Categorical Foundations for CuTe Layouts* (Colfax Research).

The previous notebook (`03_nested_tuples_and_nest.ipynb`) treated refinements of
nested tuples as a *relation* and pulled morphisms back (or pushed them forward) along
them. This notebook repackages refinements as *morphisms in their own right*:

1. the **factorization category** $\mathbf{Fact}$, whose morphisms are flat
   refinements recorded by their relative modes;
2. the **refinement category** $\mathbf{Ref}$, whose morphisms are refinements
   presented by nested tuples, remembering the full *tower* of refinements under
   composition where $\mathbf{Fact}$ flattens it;
3. **span** and **cospan** categories over $\mathbf{Fact}$ and $\mathbf{Ref}$, in which
   a Tuple morphism and a refinement are combined into a single composable morphism;
4. the **flattening functors** $\mathbf{Ref} \to \mathbf{Fact}$,
   $\mathrm{Span}(\mathbf{Tuple}, \mathbf{Ref}) \to \mathrm{Span}(\mathbf{Tuple}, \mathbf{Fact})$,
   and their cospan analogues.

In [1]:
from tract import (
    NestedTuple,
    TupleMorphism,
    FactMorphism,
    RefMorphism,
    SpanMorphism,
    CoSpanMorphism,
    RefSpanMorphism,
    RefCoSpanMorphism,
)

## 1. The category $\mathbf{Fact}$

$\mathbf{Fact}$ has

- **objects**: flat tuples of positive integers, and
- **morphisms**: a morphism $F\colon (u_1,\ldots,u_p) \to (t_1,\ldots,t_n)$ is a tuple
  of **modes** $(F_1,\ldots,F_n)$, where each $F_i$ is a flat tuple of positive
  integers with $\prod F_i = t_i$, and the concatenation of $F_1,\ldots,F_n$ equals
  $(u_1,\ldots,u_p)$.

In other words, a morphism partitions its domain into consecutive blocks, one per
codomain entry, with each block multiplying to that entry: it is precisely a *flat
refinement* $(u_1,\ldots,u_p) \twoheadrightarrow (t_1,\ldots,t_n)$ recorded by its
relative modes. Morphisms in $\mathbf{Fact}$ preserve size.

In [2]:
F = FactMorphism(domain=(2, 1, 2), codomain=(2, 2), modes=((2,), (1, 2)))
G = FactMorphism(domain=(2, 1, 2), codomain=(2, 2), modes=((2, 1), (2,)))
print(F)
print(G)
print("F == G:", F == G)
print("size preserved:", F.size() == F.cosize())

(2, 1, 2) --((2,), (1, 2))--> (2, 2)
(2, 1, 2) --((2, 1), (2,))--> (2, 2)
F == G: False
size preserved: True


Note that morphisms carry genuine data: `F` and `G` above are *distinct* morphisms
between the same objects, so $\mathbf{Fact}$ is a category, not merely the refinement
poset. The constructor validates the two defining conditions.

In [3]:
try:
    FactMorphism(domain=(2, 3), codomain=(5,), modes=((2, 3),))
except ValueError as e:
    print("ValueError:", e)

ValueError: Must satisfy prod(F_i) = t_i for all i: mode 1 has product 6, but codomain entry is 5


### Identity, composition, and sum

The identity on $(t_1,\ldots,t_n)$ is $((t_1),\ldots,(t_n))$: every mode a singleton.
Composition concatenates blocks of blocks; as everywhere in `tract`, `f.compose(g)` is
the diagrammatic composite $g \circ f$. The sum $f \oplus g$ concatenates domains,
codomains, and modes, making $\mathbf{Fact}$ monoidal.

In [4]:
f = FactMorphism((2, 2, 3, 5), (4, 15), ((2, 2), (3, 5)))   # (2,2,3,5) -> (4,15)
g = FactMorphism((4, 15), (60,), ((4, 15),))                # (4,15)    -> (60,)
print("g o f =", f.compose(g))

print("identity on (4,15):", FactMorphism.identity((4, 15)))
print("unit laws hold:",
      f.compose(FactMorphism.identity(f.codomain)) == f
      and FactMorphism.identity(f.domain).compose(f) == f)

g o f = (2, 2, 3, 5) --((2, 2, 3, 5),)--> (60,)
identity on (4,15): (4, 15) --((4,), (15,))--> (4, 15)
unit laws hold: True


In [5]:
print("f (+) g =", f.sum(g))

f (+) g = (2, 2, 3, 5, 4, 15) --((2, 2), (3, 5), (4, 15))--> (4, 15, 60)


### Relation to refinements of nested tuples

The modes of a Fact morphism assemble into a nested tuple (of depth $\leq 2$) that
refines the codomain, and every flat refinement arises this way:
`from_refinement` inverts `refined_codomain`.

In [6]:
R = f.refined_codomain()
print("modes as a nested tuple:", R)
print("refines the codomain:   ", R.refines(NestedTuple(f.codomain)))
print("round-trip:", FactMorphism.from_refinement(R, NestedTuple(f.codomain)) == f)

modes as a nested tuple: ((2,2),(3,5))
refines the codomain:    True
round-trip: True


## 2. Pullback and pushforward along a Fact morphism

A Fact morphism $b\colon T' \twoheadrightarrow T$ acts on Tuple morphisms exactly as
the refinement relation did in the previous notebook, but now with the refinement of
the *other* side returned as a Fact morphism, completing a commutative square.

**Pullback.** Given $f\colon S \to T$, `b.pullback_with_refinement(f)` returns the
pair $(r, f')$ in

$$\begin{array}{ccc}
S' & \xrightarrow{\;f'\;} & T' \\
\downarrow r & & \downarrow b \\
S & \xrightarrow{\;f\;} & T
\end{array}$$

where $r\colon S' \twoheadrightarrow S$ refines each domain entry $s_i$ with
$\alpha(i) = j \neq *$ by the mode $F_j$, and leaves basepoint entries untouched.

In [7]:
b = FactMorphism(domain=(2, 2, 3, 2), codomain=(4, 3, 2), modes=((2, 2), (3,), (2,)))
f1 = TupleMorphism(domain=(4, 3), codomain=(4, 3, 2), map=(1, 2))

r, f1_pulled = b.pullback_with_refinement(f1)
print("b         =", b)
print("f1        =", f1)
print("r         =", r)
print("pullback  =", f1_pulled)

b         = (2, 2, 3, 2) --((2, 2), (3,), (2,))--> (4, 3, 2)
f1        = (4, 3) --(1, 2)--> (4, 3, 2)
r         = (2, 2, 3) --((2, 2), (3,))--> (4, 3)
pullback  = (2, 2, 3) --(1, 2, 3)--> (2, 2, 3, 2)


In [8]:
# The square commutes on underlying maps: both composites S' -> T agree
# entrywise on where each refined entry lands.
print("legs compose correctly:",
      f1_pulled.domain == r.domain and f1_pulled.codomain == b.domain)

legs compose correctly: True


**Pushforward.** Dually, given $g\colon U \to V$ and $b\colon U' \twoheadrightarrow U$,
`b.pushforward_with_refinement(g)` returns the pair $(r, g')$ in

$$\begin{array}{ccc}
U' & \xrightarrow{\;g'\;} & V' \\
\downarrow b & & \downarrow r \\
U & \xrightarrow{\;g\;} & V
\end{array}$$

where $r\colon V' \twoheadrightarrow V$ refines each codomain entry in the image of
$g$ by the mode over its preimage, and leaves the rest of $V$ unchanged.

In [9]:
b2 = FactMorphism(domain=(4, 3, 2), codomain=(12, 2), modes=((4, 3), (2,)))
g1 = TupleMorphism(domain=(12, 2), codomain=(12, 2, 5), map=(1, 2))

r2, g1_pushed = b2.pushforward_with_refinement(g1)
print("b2          =", b2)
print("g1          =", g1)
print("r2          =", r2)
print("pushforward =", g1_pushed)

b2          = (4, 3, 2) --((4, 3), (2,))--> (12, 2)
g1          = (12, 2) --(1, 2)--> (12, 2, 5)
r2          = (4, 3, 2, 5) --((4, 3), (2,), (5,))--> (12, 2, 5)
pushforward = (4, 3, 2) --(1, 2, 3)--> (4, 3, 2, 5)


## 3. The category $\mathbf{Ref}$

$\mathbf{Fact}$ forgets *how* a refinement was built: composing two flat refinements
just merges their blocks. The category $\mathbf{Ref}$ remembers. Its objects are again
flat tuples of positive integers, and a morphism is **presented by a nested tuple**
$X = (X_1,\ldots,X_r)$, regarded as the refinement

$$ \mathrm{flat}(X) \twoheadrightarrow \rho(X) = (\mathrm{size}(X_1),\ldots,\mathrm{size}(X_r)) $$

from the flattening of $X$ to its **depth-1 reduction** $\rho(X)$, the flat tuple of
top-level mode products. The nesting of $X$ is the morphism datum: top-level mode $i$
records the factorization *tree* of the $i$-th codomain entry.

In [10]:
u = RefMorphism(((2, 3), 4))
print(u)
print("nest:    ", u.nest)
print("domain:  ", u.domain)     # flat(X)
print("codomain:", u.codomain)   # rho(X)

(2, 3, 4) --((2,3),4)--> (6, 4)
nest:     ((2,3),4)
domain:   (2, 3, 4)
codomain: (6, 4)


### Composition by grafting

The identity on $(t_1,\ldots,t_r)$ is the flat nested tuple $(t_1,\ldots,t_r)$. For
$f\colon U \twoheadrightarrow T$ and $g\colon T \twoheadrightarrow S$, the leaves of
$g$'s nested tuple are the entries of $T = \rho(f.\mathrm{nest})$, and the composite
**grafts** the $i$-th top-level mode of $f$ (as a subtree) onto the $i$-th leaf of $g$.
The composite is a *deeper* tree recording the full tower of refinements — where
$\mathbf{Fact}$ would flatten the tower into a single block structure.

In [11]:
u = RefMorphism(((2, 3), 4))      # (2,3,4) ->> (6,4)
v = RefMorphism(((6, 4),))        # (6,4)   ->> (24,)
uv = u.compose(v)                 # v o u : (2,3,4) ->> (24,)
print("u     =", u)
print("v     =", v)
print("v o u =", uv)
print("composite nest depth:", uv.nest.depth())

u     = (2, 3, 4) --((2,3),4)--> (6, 4)
v     = (6, 4) --((6,4))--> (24,)
v o u = (2, 3, 4) --(((2,3),4))--> (24,)
composite nest depth: 3


In [12]:
print("identity on (6,4):", RefMorphism.identity((6, 4)))
print("unit laws hold:",
      u.compose(RefMorphism.identity(u.codomain)) == u
      and RefMorphism.identity(u.domain).compose(u) == u)
print("u (+) v =", u.sum(v))

identity on (6,4): (6, 4) --(6,4)--> (6, 4)
unit laws hold: True
u (+) v = (2, 3, 4, 6, 4) --((2,3),4,(6,4))--> (6, 4, 24)


### The flattening functor $\mathbf{Ref} \to \mathbf{Fact}$

Fact morphisms are exactly the Ref morphisms whose top-level modes are flat (nested
tuples of depth $\leq 2$), and flattening each top-level mode is a functor
$\mathbf{Ref} \to \mathbf{Fact}$ (`to_fact_morphism`): it is the identity on objects
and preserves identities, composition, and sums. In the other direction,
`from_fact_morphism` presents a Fact morphism as a depth $\leq 2$ Ref morphism, a
section of the functor.

In [13]:
print("u as Fact:      ", u.to_fact_morphism())
print("(v o u) as Fact:", uv.to_fact_morphism())

print("functoriality:",
      uv.to_fact_morphism() == u.to_fact_morphism().compose(v.to_fact_morphism()))

u as Fact:       (2, 3, 4) --((2, 3), (4,))--> (6, 4)
(v o u) as Fact: (2, 3, 4) --((2, 3, 4),)--> (24,)
functoriality: True


In [14]:
F = FactMorphism((2, 2, 3, 5), (4, 15), ((2, 2), (3, 5)))
print("F back in Ref:", RefMorphism.from_fact_morphism(F))
print("round-trip:   ", RefMorphism.from_fact_morphism(F).to_fact_morphism() == F)

F back in Ref: (2, 2, 3, 5) --((2,2),(3,5))--> (4, 15)
round-trip:    True


Note the information loss: the tower $(2,3,4) \twoheadrightarrow (6,4)
\twoheadrightarrow (24)$ and the one-step refinement presented by the flat nest
$((2,3,4),)$ have the *same* underlying Fact morphism but are distinct in
$\mathbf{Ref}$.

In [15]:
one_step = RefMorphism(((2, 3, 4),))
print("tower:    ", uv)
print("one step: ", one_step)
print("equal in Ref: ", uv == one_step)
print("equal in Fact:", uv.to_fact_morphism() == one_step.to_fact_morphism())

tower:     (2, 3, 4) --(((2,3),4))--> (24,)
one step:  (2, 3, 4) --((2,3,4))--> (24,)
equal in Ref:  False
equal in Fact: True


### Refinements and pullback/pushforward in $\mathbf{Ref}$

As in $\mathbf{Fact}$, the presenting nested tuple refines the codomain
(`refined_codomain`), and `from_refinement` builds a Ref morphism from any refinement
$X \twoheadrightarrow T$ with $T$ flat — now the relative modes may themselves be
nested. Pullback and pushforward of Tuple morphisms work as before; the pulled-back
Tuple morphism only sees the flattened modes, but the returned refinement legs are Ref
morphisms carrying the subtrees.

In [16]:
refined = NestedTuple(((2, (3, 2)), 5))
coarse  = NestedTuple((12, 5))
w = RefMorphism.from_refinement(refined, coarse)
print("w =", w)
print("refined codomain:", w.refined_codomain())

w = (2, 3, 2, 5) --((2,(3,2)),5)--> (12, 5)
refined codomain: ((2,(3,2)),5)


In [17]:
t = TupleMorphism(domain=(12, 7), codomain=(12, 5), map=(1, 0))
r_ref, t_pulled = w.pullback_with_refinement(t)
print("t        =", t)
print("r (Ref)  =", r_ref)
print("pullback =", t_pulled)

t        = (12, 7) --(1, 0)--> (12, 5)
r (Ref)  = (2, 3, 2, 7) --((2,(3,2)),7)--> (12, 7)
pullback = (2, 3, 2, 7) --(1, 2, 3, 0)--> (2, 3, 2, 5)


## 4. Spans and cospans

The pullback and pushforward squares suggest a single category in which a refinement
and a Tuple morphism compose as one arrow. All four span/cospan categories in `tract`
share one construction: objects are flat tuples of positive integers, and a morphism
$U \to V$ is a **span**

$$ U \xleftarrow{\;b\;} X \xrightarrow{\;f\;} V $$

with backward leg $b\colon X \twoheadrightarrow U$ in the refinement category
($\mathbf{Fact}$ or $\mathbf{Ref}$) and forward leg $f\colon X \to V$ a Tuple morphism,
sharing the **apex** $X$ — or the mirror-image **cospan**

$$ U \xrightarrow{\;f\;} X \xleftarrow{\;b\;} V $$

with the backward leg out of the **nadir** $X$. Concretely:

| class | category |
|---|---|
| `SpanMorphism` | $\mathrm{Span}(\mathbf{Tuple}, \mathbf{Fact})$ |
| `CoSpanMorphism` | $\mathrm{CoSpan}(\mathbf{Tuple}, \mathbf{Fact})$ |
| `RefSpanMorphism` | $\mathrm{Span}(\mathbf{Tuple}, \mathbf{Ref})$ |
| `RefCoSpanMorphism` | $\mathrm{CoSpan}(\mathbf{Tuple}, \mathbf{Ref})$ |

In [18]:
# (12,) <--((4,3),)-- (4,3) --(1,2)--> (4,3,2)
b1 = FactMorphism(domain=(4, 3), codomain=(12,), modes=((4, 3),))
f1 = TupleMorphism(domain=(4, 3), codomain=(4, 3, 2), map=(1, 2))
s1 = SpanMorphism(b1, f1)
print(s1)
print("domain:", s1.domain, "  apex:", s1.apex, "  codomain:", s1.codomain)

(12,) <--((4, 3),)-- (4, 3) --(1, 2)--> (4, 3, 2)
domain: (12,)   apex: (4, 3)   codomain: (4, 3, 2)


### Composition via pullback

To compose $U \xleftarrow{b_1} X \xrightarrow{f_1} V$ with
$V \xleftarrow{b_2} Y \xrightarrow{f_2} W$, we pull $f_1$ back along $b_2$
(`pullback_with_refinement`), completing the square with a new corner $X'$:

$$\begin{array}{ccccccc}
X' & \xrightarrow{f_1'} & Y & \xrightarrow{f_2} & W \\
\downarrow r & & \downarrow b_2 \\
X & \xrightarrow{f_1} & V \\
\downarrow b_1 \\
U
\end{array}$$

The composite is $U \xleftarrow{b_1 \circ r} X' \xrightarrow{f_2 \circ f_1'} W$.
Because the chosen pullback substitutes each apex entry by its refining block (with
basepoint entries passing through untouched), composition is strictly associative and
unital: $\mathrm{Span}(\mathbf{Tuple}, \mathbf{Fact})$ is a genuine 1-category, not
just a bicategory.

In [19]:
# (4,3,2) <--((2,2),(3,),(2,))-- (2,2,3,2) --(1,2,0,3)--> (2,2,2)
b2 = FactMorphism((2, 2, 3, 2), (4, 3, 2), ((2, 2), (3,), (2,)))
f2 = TupleMorphism((2, 2, 3, 2), (2, 2, 2), (1, 2, 0, 3))
s2 = SpanMorphism(b2, f2)

composite = s1.compose(s2)   # diagrammatic order: s1 then s2
print(composite)
print("apex refined from", s1.apex, "to", composite.apex)

(12,) <--((2, 2, 3),)-- (2, 2, 3) --(1, 2, 0)--> (2, 2, 2)
apex refined from (4, 3) to (2, 2, 3)


In [20]:
idV = SpanMorphism.identity(s1.codomain)
print("identity on", s1.codomain, ":", idV)
print("unit laws hold:",
      s1.compose(idV) == s1
      and SpanMorphism.identity(s1.domain).compose(s1) == s1)
print("legwise sum s1 (+) s2 has domain", s1.sum(s2).domain,
      "and codomain", s1.sum(s2).codomain)

identity on (4, 3, 2) : (4, 3, 2) <--((4,), (3,), (2,))-- (4, 3, 2) --(1, 2, 3)--> (4, 3, 2)
unit laws hold: True
legwise sum s1 (+) s2 has domain (12, 4, 3, 2) and codomain (4, 3, 2, 2, 2, 2)


### The category $\mathrm{CoSpan}(\mathbf{Tuple}, \mathbf{Fact})$

Dually, a cospan $U \xrightarrow{f} X \xleftarrow{b} V$ has a Tuple morphism as its
forward (left) leg and a Fact morphism $b\colon X \twoheadrightarrow V$ out of the
nadir as its backward (right) leg. In both constructions the refinement leg has the
apex/nadir as its *domain*: the middle object is a refinement of the boundary object it
points to.

In [21]:
# (4,3) --(1,2)--> (4,3,2) <--((4,3),(2,))-- (12,2)
cf1 = TupleMorphism((4, 3), (4, 3, 2), (1, 2))
cb1 = FactMorphism((4, 3, 2), (12, 2), ((4, 3), (2,)))
c1 = CoSpanMorphism(cf1, cb1)
print(c1)
print("domain:", c1.domain, "  nadir:", c1.nadir, "  codomain:", c1.codomain)

(4, 3) --(1, 2)--> (4, 3, 2) <--((4, 3), (2,))-- (12, 2)
domain: (4, 3)   nadir: (4, 3, 2)   codomain: (12, 2)


Composition is dual: to compose $U \xrightarrow{f_1} X \xleftarrow{b_1} V$ with
$V \xrightarrow{f_2} Y \xleftarrow{b_2} W$, we push $f_2$ forward along $b_1$
(`pushforward_with_refinement`), completing the square with a new corner $Y'$, and
compose the legs:

$$ U \xrightarrow{\;f_2' \circ f_1\;} Y' \xleftarrow{\;b_2 \circ r\;} W. $$

In [22]:
# (12,2) --(1,2)--> (12,2,5) <--((12,),(2,5))-- (12,10)
cf2 = TupleMorphism((12, 2), (12, 2, 5), (1, 2))
cb2 = FactMorphism((12, 2, 5), (12, 10), ((12,), (2, 5)))
c2 = CoSpanMorphism(cf2, cb2)

ccomposite = c1.compose(c2)   # diagrammatic order: c1 then c2
print(ccomposite)
print("nadir refined from", c2.nadir, "to", ccomposite.nadir)

idV = CoSpanMorphism.identity(c1.codomain)
print("unit laws hold:",
      c1.compose(idV) == c1
      and CoSpanMorphism.identity(c1.domain).compose(c1) == c1)

(4, 3) --(1, 2)--> (4, 3, 2, 5) <--((4, 3), (2, 5))-- (12, 10)
nadir refined from (12, 2, 5) to (4, 3, 2, 5)
unit laws hold: True


## 5. Spans and cospans over $\mathbf{Ref}$

`RefSpanMorphism` and `RefCoSpanMorphism` are the same constructions with the backward
leg a Ref morphism: the apex (resp. nadir) is presented by a nested tuple $N$ whose
flattening is the middle object and whose depth-1 reduction is the boundary object, so
the middle object carries an arbitrary *nested* factorization. Composition works
identically — pullback (resp. pushforward) around the shared middle object — but the
composite backward leg now grafts, remembering the tower of refinements accumulated
along the way.

In [23]:
# (12,) <--((4,3),)-- (4,3) --(1,2)--> (4,3,2)
rb1 = RefMorphism(((4, 3),))
rs1 = RefSpanMorphism(rb1, f1)
print(rs1)
print("domain:", rs1.domain, "  apex:", rs1.apex, "  codomain:", rs1.codomain)

(12,) <--((4,3))-- (4, 3) --(1, 2)--> (4, 3, 2)
domain: (12,)   apex: (4, 3)   codomain: (4, 3, 2)


In [24]:
# (4,3,2) <--(((2,2)),3,2)-- (2,2,3,2) --(1,2,0,3)--> (2,2,2)
# The first codomain entry 4 is refined by the *nested* mode ((2,2)).
rb2 = RefMorphism((((2, 2),), 3, 2))
rs2 = RefSpanMorphism(rb2, f2)
print(rs2)

rcomposite = rs1.compose(rs2)
print("composite:", rcomposite)
print("composite backward nest:", rcomposite.left.nest,
      " (depth", str(rcomposite.left.nest.depth()) + ")")

(4, 3, 2) <--(((2,2)),3,2)-- (2, 2, 3, 2) --(1, 2, 0, 3)--> (2, 2, 2)
composite: (12,) <--((((2,2)),3))-- (2, 2, 3) --(1, 2, 0)--> (2, 2, 2)
composite backward nest: ((((2,2)),3))  (depth 4)


The composite backward leg's nest records that $12$ was first factored as $(4,3)$ and
then $4$ was further factored — a tower the Fact-based span category would have
flattened into a single block.

In [25]:
print("identity and unit laws:",
      rs1.compose(RefSpanMorphism.identity(rs1.codomain)) == rs1
      and RefSpanMorphism.identity(rs1.domain).compose(rs1) == rs1)

identity and unit laws: True


### Bridge functors and functoriality

Flattening the backward leg gives functors
$\mathrm{Span}(\mathbf{Tuple}, \mathbf{Ref}) \to \mathrm{Span}(\mathbf{Tuple}, \mathbf{Fact})$
(`to_span_morphism`) and
$\mathrm{CoSpan}(\mathbf{Tuple}, \mathbf{Ref}) \to \mathrm{CoSpan}(\mathbf{Tuple}, \mathbf{Fact})$
(`to_cospan_morphism`), identity on objects, with sections `from_span_morphism` /
`from_cospan_morphism`. Functoriality means composing in
$\mathrm{Span}(\mathbf{Tuple}, \mathbf{Ref})$ and then flattening equals flattening
first and then composing in $\mathrm{Span}(\mathbf{Tuple}, \mathbf{Fact})$:

In [26]:
compose_then_flatten = rs1.compose(rs2).to_span_morphism()
flatten_then_compose = rs1.to_span_morphism().compose(rs2.to_span_morphism())
print("compose then flatten:", compose_then_flatten)
print("flatten then compose:", flatten_then_compose)
print("equal:", compose_then_flatten == flatten_then_compose)

compose then flatten: (12,) <--((2, 2, 3),)-- (2, 2, 3) --(1, 2, 0)--> (2, 2, 2)
flatten then compose: (12,) <--((2, 2, 3),)-- (2, 2, 3) --(1, 2, 0)--> (2, 2, 2)
equal: True


In [27]:
# Round trip through the section, and identity preservation.
print("section round-trip:",
      RefSpanMorphism.from_span_morphism(s1).to_span_morphism() == s1)
print("identities map to identities:",
      RefSpanMorphism.identity((4, 3, 2)).to_span_morphism()
      == SpanMorphism.identity((4, 3, 2)))

section round-trip: True
identities map to identities: True


The same holds on the cospan side, where composition pushes forward along a nested
backward leg.

In [28]:
# (2,3) --(1,3)--> (2,2,3,2) <--(((2,2),3),2)-- (12,2)
# The nadir presents 12 by the *nested* factorization ((2,2),3).
rcf1 = TupleMorphism((2, 3), (2, 2, 3, 2), (1, 3))
rc1 = RefCoSpanMorphism(rcf1, RefMorphism((((2, 2), 3), 2)))
# (12,2) --(1,2)--> (12,2,5) <--(12,(2,5))-- (12,10)
rc2 = RefCoSpanMorphism(cf2, RefMorphism((12, (2, 5))))

rc = rc1.compose(rc2)
print("composite:", rc)
print("composite backward nest:", rc.right.nest,
      " (depth", str(rc.right.nest.depth()) + ")")

composite: (2, 3) --(1, 3)--> (2, 2, 3, 2, 5) <--(((2,2),3),(2,5))-- (12, 10)
composite backward nest: (((2,2),3),(2,5))  (depth 3)


In [29]:
print("cospan functoriality:",
      rc1.compose(rc2).to_cospan_morphism()
      == rc1.to_cospan_morphism().compose(rc2.to_cospan_morphism()))
print("cospan section round-trip:",
      RefCoSpanMorphism.from_cospan_morphism(c1).to_cospan_morphism() == c1)

cospan functoriality: True
cospan section round-trip: True


## 6. Remark on naming

Forgetting the integer values, a $\mathbf{Fact}$ morphism
$(u_1,\ldots,u_p) \to (t_1,\ldots,t_n)$ is exactly an *active* (endpoint-preserving)
map $[p] \to [n]$ in the simplex category $\Delta$, so $\mathbf{Fact}$ is the category
of elements of the product-pushforward functor on $\Delta_{\mathrm{act}}$ over the
monoid $(\mathbb{Z}_{>0}, \times)$. No canonical name for this category appears in the
literature; `Fact` (for *factorization*) is our working name, and `Ref` its
tower-remembering thickening.

## Summary

- $\mathbf{Fact}$: flat refinements as morphisms, recorded by relative modes;
  composition merges blocks of blocks.
- $\mathbf{Ref}$: refinements presented by nested tuples
  $\mathrm{flat}(X) \twoheadrightarrow \rho(X)$; composition grafts subtrees,
  remembering the tower of refinements; flattening top-level modes is a functor
  $\mathbf{Ref} \to \mathbf{Fact}$.
- Both categories act on Tuple morphisms by pullback and pushforward, returning the
  commutative square's fourth side as a refinement in the same category.
- $\mathrm{Span}(\mathbf{Tuple}, -)$ and $\mathrm{CoSpan}(\mathbf{Tuple}, -)$ package
  a refinement and a Tuple morphism into one arrow; composition goes around the shared
  apex/nadir via the chosen pullback/pushforward and is strictly associative and
  unital.
- Flattening the backward legs gives bridge functors from the Ref-based span and
  cospan categories to the Fact-based ones, verified here on explicit composites.